# 🤖 RoboQuest 2026 — ロボットを鬼ごっこで勝たせよう！

このノートブックでは **スライダーを動かすだけ** でロボットに「鬼から逃げる」動きを学習させられます。

**手順:**
1. ▶ ボタンを押してセルを上から順番に実行してください
2. 「🤖 ロボットのパラメータ設定」のスライダーで数値を変えてみよう
3. 「🚀 学習スタート！」を実行して待つ
4. 「📊 結果を見てみよう」で動画を確認！

---
> 💡 **ヒント:** スライダーの値を変えると、ロボットの逃げ方が変わります。
> どんな値にしたら一番うまく逃げられるか、チームで考えてみよう！

In [ ]:
#@title 🔧 セットアップ（最初に一度だけ実行してください）

import subprocess, sys, os

# ── 1. GPU/CPU を自動検出してレンダリングバックエンドを設定 ──────────────────
#    mujoco を import する前に MUJOCO_GL を設定する必要がある
_has_gpu = subprocess.run('nvidia-smi', shell=True, capture_output=True).returncode == 0

if _has_gpu:
    # GPU あり → NVIDIA EGL（ハードウェアアクセラレーション）
    NVIDIA_ICD_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
    if not os.path.exists(NVIDIA_ICD_PATH):
        os.makedirs(os.path.dirname(NVIDIA_ICD_PATH), exist_ok=True)
        with open(NVIDIA_ICD_PATH, 'w') as _f:
            _f.write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}\n')
    os.environ['MUJOCO_GL'] = 'egl'
    print('🖥️  GPU 検出: EGL レンダリングを使用します')
else:
    # CPU のみ → OSMesa（ソフトウェアレンダリング、GPU 不要）
    subprocess.run('apt-get install -y -q libosmesa6', shell=True, check=False)
    os.environ['MUJOCO_GL'] = 'osmesa'
    print('💻  CPU モード: OSMesa レンダリングを使用します')

# ffmpeg（動画保存に必要）
subprocess.run(
    'command -v ffmpeg >/dev/null || (apt-get update -qq && apt-get install -y -q ffmpeg)',
    shell=True, check=False)

# ── 2. Python ライブラリのインストール ──────────────────────────────────────
print('ライブラリをインストール中...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'mujoco', 'gymnasium', 'stable-baselines3', 'mediapy', 'tqdm', 'pandas', 'matplotlib'],
    check=True)

# ── 3. リポジトリのクローン ─────────────────────────────────────────────────
if not os.path.exists('/content/RoboQuest2026'):
    print('リポジトリをダウンロード中...')
    subprocess.run(['git', 'clone', '-q',
        'https://github.com/tadryo/RoboQuest2026.git',
        '/content/RoboQuest2026'], check=True)

os.chdir('/content/RoboQuest2026')
if '/content/RoboQuest2026' not in sys.path:
    sys.path.insert(0, '/content/RoboQuest2026')

# ── 4. Go2 モデルのダウンロード ─────────────────────────────────────────────
print('Go2 ロボットのモデルをダウンロード中...')
subprocess.run([sys.executable, 'scripts/download_models.py'], check=True)

print('\n✅ セットアップ完了！次のセルへ進んでください。')


In [ ]:
#@title 📁 チーム名と Google Drive 接続

#@markdown チーム名を入力してください（モデルの保存先になります）
team_name = '自分のチーム名' #@param {type:"string"}

from google.colab import drive
print('Google Drive に接続します...')
drive.mount('/content/drive')

import os
SAVE_DIR = f'/content/drive/MyDrive/RoboQuest2026/{team_name}'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'\n✅ 保存先: {SAVE_DIR}')

In [ ]:
#@title 🤖 ロボットのパラメータ設定

#@markdown ## 🏃 逃げ方の設定
#@markdown ---

#@markdown **鬼から離れる重み** — 大きいほど「とにかく鬼から離れる」を優先
distance_weight = 1.0 #@param {type:"slider", min:0.0, max:3.0, step:0.1}

#@markdown **生存ボーナスの重み** — 大きいほど「慎重に生き延びる」を優先
survival_weight = 0.1 #@param {type:"slider", min:0.0, max:1.0, step:0.05}

#@markdown **エネルギー効率の重み** — 大きいほど「省エネ・滑らかな動き」に
control_weight = 0.05 #@param {type:"slider", min:0.0, max:0.5, step:0.01}

#@markdown **前進重み** — 大きいほど「直線的に逃げる」を優先
forward_weight = 0.5 #@param {type:"slider", min:0.0, max:2.0, step:0.1}

#@markdown ---
#@markdown 💡 **考えてみよう:** どの数値を大きくしたら鬼から一番うまく逃げられる？

print(f'パラメータ設定:')
print(f'  鬼から離れる重み: {distance_weight}')
print(f'  生存ボーナス: {survival_weight}')
print(f'  エネルギー効率: {control_weight}')
print(f'  前進重み: {forward_weight}')

In [ ]:
#@title ⚙️ 学習の設定

#@markdown **学習ステップ数** — 大きいほど長く学習する（時間がかかる）
total_timesteps = 300000 #@param {type:"slider", min:50000, max:1000000, step:50000}

#@markdown **学習率** — ニューラルネットワークの更新速度（通常は変えなくてOK）
learning_rate = 0.0003 #@param {type:"slider", min:0.0001, max:0.001, step:0.0001}

#@markdown **並列環境数** — 同時にシミュレーションする数（多いと速いが重い）
num_envs = 4 #@param {type:"slider", min:1, max:16, step:1}

print(f'学習設定:')
print(f'  学習ステップ数: {total_timesteps:,}')
print(f'  学習率: {learning_rate}')
print(f'  並列環境数: {num_envs}')
print(f'\n⏱ 学習時間の目安: 約 {total_timesteps // 100000 * 5} 〜 {total_timesteps // 100000 * 15} 分')

In [ ]:
#@title 🚀 学習スタート！

import os, sys
os.chdir('/content/RoboQuest2026')
if '/content/RoboQuest2026' not in sys.path:
    sys.path.insert(0, '/content/RoboQuest2026')

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import CheckpointCallback

from roboquest.envs.go2_tag_env import Go2TagEnv
from roboquest.utils.reward_utils import RewardConfig

# 報酬設定
reward_cfg = RewardConfig(
    distance_weight=distance_weight,
    survival_weight=survival_weight,
    control_weight=control_weight,
    forward_weight=forward_weight,
)

# 環境作成
def make_env(rank=0):
    def _init():
        env = Go2TagEnv(reward_config=reward_cfg)
        return Monitor(env, filename=f'/tmp/monitor_{rank}')
    return _init

train_env = SubprocVecEnv([make_env(i) for i in range(num_envs)])
train_env = VecNormalize(train_env, norm_obs=True, norm_reward=True)

# モデル作成
model = PPO(
    'MlpPolicy',
    train_env,
    learning_rate=learning_rate,
    n_steps=1024,
    batch_size=256,
    n_epochs=10,
    policy_kwargs={'net_arch': [256, 256, 128]},
    verbose=0,
)

print(f'=== 学習開始 ===')
print(f'合計ステップ数: {total_timesteps:,}')
print(f'設定: distance={distance_weight}, survival={survival_weight}, '
      f'control={control_weight}, forward={forward_weight}')
print('学習中... しばらくお待ちください\n')

model.learn(
    total_timesteps=total_timesteps,
    progress_bar=True,
)

# モデル保存（Drive + ローカル）
LOCAL_SAVE = '/tmp/flee_model'
model.save(LOCAL_SAVE)
train_env.save('/tmp/vec_normalize.pkl')

try:
    model.save(f'{SAVE_DIR}/flee_final')
    train_env.save(f'{SAVE_DIR}/vec_normalize.pkl')
    print(f'\n✅ 学習完了！モデルを保存しました: {SAVE_DIR}')
except Exception:
    print(f'\n✅ 学習完了！モデルをローカルに保存しました: {LOCAL_SAVE}.zip')

In [ ]:
import os
# Colab ヘッドレス環境: mujoco より前に EGL を設定（必須）
os.environ["MUJOCO_GL"] = "egl"

#@title 📊 結果を見てみよう

import os, sys, glob
os.chdir('/content/RoboQuest2026')
if '/content/RoboQuest2026' not in sys.path:
    sys.path.insert(0, '/content/RoboQuest2026')

import numpy as np
import matplotlib.pyplot as plt
import mujoco

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from roboquest.envs.go2_tag_env import Go2TagEnv

# --- 学習曲線 ---
print('📈 学習曲線を表示します...')
monitor_files = glob.glob('/tmp/monitor_*.monitor.csv')

if monitor_files:
    import pandas as pd
    dfs = []
    for f in monitor_files:
        try:
            df = pd.read_csv(f, skiprows=1)
            dfs.append(df)
        except Exception:
            pass
    if dfs:
        df_all = pd.concat(dfs).sort_values('t').reset_index(drop=True)
        rewards = df_all['r'].values
        w = min(50, len(rewards))
        smooth = np.convolve(rewards, np.ones(w)/w, mode='valid')

        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(rewards, alpha=0.3, color='steelblue', label='エピソード報酬')
        ax.plot(np.arange(w-1, len(rewards)), smooth,
                color='steelblue', lw=2, label=f'移動平均')
        ax.set_xlabel('エピソード数')
        ax.set_ylabel('報酬')
        ax.set_title('学習の進み具合')
        ax.legend()
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()
        print(f'最終 20 エピソードの平均報酬: {np.mean(rewards[-20:]):.2f}')
else:
    print('学習ログが見つかりません')

# --- 鬼ごっこ動画 ---
print('\n🎬 学習後のロボットの動きを録画します...')

loaded_model = PPO.load('/tmp/flee_model')
env = Go2TagEnv(render_mode='rgb_array')

renderer = mujoco.Renderer(env.model, height=480, width=640)
frames = []
obs, _ = env.reset()
survival_steps = 0
distances = []

for _ in range(600):
    action, _ = loaded_model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    distances.append(info.get('oni_distance', 0))
    survival_steps += 1

    renderer.update_scene(env.data)
    frames.append(renderer.render().copy())

    if terminated or truncated:
        break

renderer.close()
env.close()

# 動画を表示
try:
    import mediapy as media
    media.show_video(frames, fps=30)
except ImportError:
    from matplotlib import animation
    from IPython.display import HTML, display
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.axis('off')
    img = ax.imshow(frames[0])
    ani = animation.FuncAnimation(
        fig, lambda f: [img.set_array(f)], frames=frames, interval=33, blit=True)
    plt.close(fig)
    display(HTML(ani.to_jshtml()))

print(f'\n📊 結果サマリー')
print(f'  生存ステップ数: {survival_steps} ステップ')
print(f'  鬼との平均距離: {np.mean(distances):.2f} m')
print(f'  鬼との最大距離: {np.max(distances):.2f} m')
captured = info.get('is_tagged', 0)
print(f'  タグされたか: {"はい" if captured else "いいえ（逃げきり！）"}')

In [ ]:
#@title 💾 モデルを保存して大会に提出

import os, yaml

# パラメータを記録
params = {
    'team_name': team_name,
    'distance_weight': distance_weight,
    'survival_weight': survival_weight,
    'control_weight': control_weight,
    'forward_weight': forward_weight,
    'total_timesteps': total_timesteps,
    'learning_rate': learning_rate,
    'survival_steps': survival_steps,
    'avg_distance': float(np.mean(distances)),
}

try:
    with open(f'{SAVE_DIR}/params.yaml', 'w', encoding='utf-8') as f:
        yaml.dump(params, f, allow_unicode=True)
    print(f'✅ パラメータを保存しました: {SAVE_DIR}/params.yaml')
    print(f'✅ モデルを保存しました: {SAVE_DIR}/flee_final.zip')
    print(f'\n🎉 大会への提出準備完了！')
    print(f'   保存先: {SAVE_DIR}')
except Exception as e:
    print(f'Drive 保存でエラーが発生しました: {e}')
    print('ローカル保存済み: /tmp/flee_model.zip')